In [24]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

In [25]:
csv_path = "./titanic.csv"

In [26]:
# start spark session 
spark = SparkSession.builder.appName('Example').getOrCreate()
spark

In [27]:
titanic_df = spark.read.csv(path=csv_path, header=True, inferSchema=True)
titanic_df.show(5)

+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+
|Survived|Pclass|                Name|   Sex| Age|Siblings/Spouses Aboard|Parents/Children Aboard|   Fare|
+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+
|       0|     3|Mr. Owen Harris B...|  male|22.0|                      1|                      0|   7.25|
|       1|     1|Mrs. John Bradley...|female|38.0|                      1|                      0|71.2833|
|       1|     3|Miss. Laina Heikk...|female|26.0|                      0|                      0|  7.925|
|       1|     1|Mrs. Jacques Heat...|female|35.0|                      1|                      0|   53.1|
|       0|     3|Mr. William Henry...|  male|35.0|                      0|                      0|   8.05|
+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+
only showing top 5 rows



In [28]:
assembler = VectorAssembler(inputCols=['Pclass', 'Fare','Age'], outputCol='Independent Features')

In [29]:
output = assembler.transform(titanic_df)

In [30]:
output.show(5)

+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+--------------------+
|Survived|Pclass|                Name|   Sex| Age|Siblings/Spouses Aboard|Parents/Children Aboard|   Fare|Independent Features|
+--------+------+--------------------+------+----+-----------------------+-----------------------+-------+--------------------+
|       0|     3|Mr. Owen Harris B...|  male|22.0|                      1|                      0|   7.25|     [3.0,7.25,22.0]|
|       1|     1|Mrs. John Bradley...|female|38.0|                      1|                      0|71.2833|  [1.0,71.2833,38.0]|
|       1|     3|Miss. Laina Heikk...|female|26.0|                      0|                      0|  7.925|    [3.0,7.925,26.0]|
|       1|     1|Mrs. Jacques Heat...|female|35.0|                      1|                      0|   53.1|     [1.0,53.1,35.0]|
|       0|     3|Mr. William Henry...|  male|35.0|                      0|                      0|   8.0

In [31]:
finalized_data = output.select('Independent Features', 'Survived')

In [32]:
finalized_data.show(5)

+--------------------+--------+
|Independent Features|Survived|
+--------------------+--------+
|     [3.0,7.25,22.0]|       0|
|  [1.0,71.2833,38.0]|       1|
|    [3.0,7.925,26.0]|       1|
|     [1.0,53.1,35.0]|       1|
|     [3.0,8.05,35.0]|       0|
+--------------------+--------+
only showing top 5 rows



In [34]:
train_data, test_data = finalized_data.randomSplit([0.75, 0.25])
model = LogisticRegression(featuresCol='Independent Features', labelCol='Survived')
model = model.fit(train_data)

24/09/24 15:56:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/09/24 15:56:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [ ]:
model.

In [39]:
predictions = model.transform(test_data)
predictions

DataFrame[Independent Features: vector, Survived: int, rawPrediction: vector, probability: vector, prediction: double]

In [41]:
predictions.drop('rawPrediction').show()

+--------------------+--------+--------------------+----------+
|Independent Features|Survived|         probability|prediction|
+--------------------+--------+--------------------+----------+
|      [1.0,0.0,39.0]|       0|[0.40069164858809...|       1.0|
|  [1.0,25.5875,47.0]|       0|[0.44043002605778...|       1.0|
|  [1.0,25.9292,48.0]|       1|[0.44786417610689...|       1.0|
|  [1.0,25.9292,49.0]|       1|[0.45560996764668...|       1.0|
|     [1.0,26.0,57.0]|       0|[0.51797525912388...|       0.0|
|  [1.0,26.2833,19.0]|       1|[0.24647721111967...|       1.0|
|    [1.0,26.55,28.0]|       1|[0.30218387132847...|       1.0|
|    [1.0,26.55,41.0]|       0|[0.39404462412790...|       1.0|
|  [1.0,27.7208,44.0]|       1|[0.41568827827747...|       1.0|
|     [1.0,29.7,37.0]|       0|[0.36212390755442...|       1.0|
|     [1.0,29.7,58.0]|       0|[0.52263997791004...|       0.0|
|     [1.0,30.0,29.0]|       0|[0.30632014717712...|       1.0|
|     [1.0,30.5,27.0]|       1|[0.292839